#### Vector Databases for Semantic Search

- In LangChain, vector databases (referred to as Vector Stores) serve as the foundational backbone for Retrieval-Augmented Generation (RAG) and semantic search workflows. 
- They are specialized storage engines designed to store, manage, and index high-dimensional numeric arrays called vector embeddings, which represent the underlying semantic meaning of unstructured text data.
- Instead of performing traditional keyword matches, LangChain integrates with these databases to run low-latency similarity searches. This lets LLM applications dynamically fetch contextually relevant document snippets based on the intent behind a user's prompt.

#### Why we need the Vecotr Stores.
- suppose we have a 10k chunks from pdfs. 
- example : we have the below chunks 
```
Chunk 1 : Spark is a distributed processing engine.

Chunk 2 : Kafka supports event streaming.

Chunk 3 : Airflow is an orchestration tool.
```

- after embedding : there is should be something to store right.
```
Chunk 1
[0.21,0.56,-0.88,...]

Chunk 2
[-0.44,0.91,0.32,...]

Chunk 3
[0.78,-0.15,0.67,...]
```
- where should we store this vecotr basicall? certailnly in python list() and dictionaries() where searching millions of vectors becomes very slow. for that we need a specialized database for this we have the vector database.

- vecotr database stores embeddings and allow semantic similarity search.


##### Tradtion Databases vs Vector Databases
- Sql Database : if we search for here one key word like this   ```WHERE name = 'Spark'```, here we are doing the exact match
- Vector Database: search ```"distributed data processing"``` may retrive: ```ApacheSpark``` becuse the meaning is similar this is called *semantic search*


##### Why not key word match 
- let us say we have document : Dogs are loyal animals. here if user asks ```tell me about puppies``` here if we do key word search: ```dogs ≠ puppies``` so it will become no match.
- embedding search understands ```dogs ≈ puppies``` and retrives the correct chunk.




#### What Does a Vector Store Contain?
- A vector store contains mathematical representations of unstructured data called vector embeddings
- These embeddings translate complex data (like text, images, or audio) into long arrays of floating-point numbers that AI models can process and compare.
- Specifically, each "record" or "point" stored in a vector database contains three main components:
    1. *Vector Embeddings*: The actual arrays of numbers (dimensions) representing the data. Items with similar meanings are mapped closer together in this mathematical space, allowing AI to find context rather than exact keyword matches.
    2. *Metadata* : Key-value pairs of descriptive information attached to the vector. this can include data like the source document title, creation date, author, tags which are used to filter the search result.
    3. Unique Identifier (ID): A specific string or number used to track and retrieve that exact record.

##### Common Vector Databases 
- Faiss : Facebook AI similarity Search, it runs locally
- Chroma : Simple and begineer friendly, persistent storage.
- pine cone : Managed cloud Vector Db. and this is the most common common database used in the productions.
- weavite : open source vector database
- quadrant : Fast and Scalable
- Milvus : Entrprice sclae
- Elastic Search: Supports vector search 

##### Similarity Search
- Similarity search is a method used by AI databases to find data points that are conceptually or contextually close to each other. 
-  Instead of matching exact keywords (like a traditional search engine), it calculates the mathematical distance between vector embeddings to retrieve the most relevant information

##### How Similarity Search Works 
- The process transforms human queries into mathemetical coordinates to locate the closest matches in a multi-dimensional space.
```
[User Query] ──> [Embedding Model] ──> [Query Vector]
                                             │
                                             ▼
                                  ┌─────────────────────┐
                                  │  Distance Metrics:  │
                                  │  • Cosine           │
                                  │  • Euclidean        │
                                  │  • Dot Product      │
                                  └──────────┬──────────┘
                                             │
                                             ▼
                                [Nearest Neighbor Matches]
```

1. Vectorizing the Query: when a user submits a query (example: "Vehicles that fly"), it is passed to the same embedding model used to build the database. the model translates the query text into high dimensional vector(a string of numbers)
2. Measuring the Mathematical Distance: The vecotr store compares the query vector against all stored vectors. it mesures the "Distance" or angle between them using specific algorithms called *Distance Metrics*.
    - *Cosine Similarity* : Measures the angle between two vectors. it focuses on the direction rather than length, making it ideal for determining it two pieces of text share the same topic.
    - *Eucidean Distance* : Measures the straight-line distance between two coordinates in space. smaller distances mean higher similarity.
    - *Dot Product* : Multiplies corresponding coordinates and sum them up. it evaulates the both the angles and the magnitude(length) of the vectors.
3. Retriving the nearest neighbors:
    - the database ranks the stored records based on the distance scores. it returns the TOP K results with the smallest distance (highest similarity) to the users query. This is known as the *K-Nearest Neibhors(KNN) Approach*

#### Understanding Cosine Similarity
To see how this works mathematically, look at how the angle θ determines similarity. When two vectors point in the same direction, the angle θ = 0°, and \(\cos(0^\circ) = 1\) (perfect similarity). If they are completely unrelated (perpendicular), θ = 90°, and \(\cos(90^\circ) = 0\)

![alt text](image.png)

#### Keyword Search vs. Similarity Search
| Feature | Keyword Search (Lexical) | Similarity Search (Semantic) |
| :--- | :--- | :--- |
| **Matching Mechanism** | Exact character strings | Contextual meanings and concepts |
| **Synonym Handling** | Fails unless hardcoded | Automatic (maps "car" near "automobile") |
| **Data Types** | Primarily text | Text, images, audio, video |
| **Example Match** | "Apple phone" &rarr; "Apple phone" | "Apple phone" &rarr; "iPhone 15 specs" |


#### k Parameter
- The K paramter represents the exact number of closest matches. (the "nearest neighbors") the database should return for our query.
- it acts as a limit or cutoff for your search result.

#### Key Details About the K Parameter
- Explicit Limit: If you set K = 5, the vector store will calculate the mathematical distances, rank all items, and return only the top 5 most similar records.
- The "KNN" Name: This parameter gives the K-Nearest Neighbors (KNN) algorithm its name
- Balance of Information:
    1. Setting K too low (e.g., K=1) might miss helpful background context.
    2. Setting K too high (e.g., K=50) can introduce irrelevant data and overwhelm the LLM's context window.

##### Where it fits in a code
- When writing code to query a vector database, you will usually pass k (sometimes called top_k) as a variable:
##### Example query looking for the top 3 closest matches
results = vector_store.similarity_search(
    query="How do planes fly?", 
    k=3
)


#### Why we Cannot Mix Models
- we must use the exact same embedding model for both document ingestion and querying. If you use different models, your search will fail completely and return irrelevant results.
- An embedding model creates its own unique, multi-dimensional mathematical "language map."
    - Different Dimensions: One model might convert a sentence into an array of 768 numbers (dimensions), while another converts it into 1536 numbers. You cannot mathematically compare vectors of different lengths.
    - Different Meaning Maps: Even if two models use the exact same number of dimensions, they map concepts differently.
        - Model A might use coordinate position 42 to represent the concept of "flying."
        - Model B might use coordinate position 42 to represent "color."
    - If you embed your documents with Model A and query with Model B, the coordinates will point to random, unrelated spaces in the database.

#### Are All Vector Embeddings the Same?
- No, vector embeddings vary significantly between models. They are not standardized universal numbers.
- Models differ based on several critical design factors:
    - Dimensional Size: Models like text-embedding-3-small output 1,536 dimensions, while open-source models like BGE-large output 1,024 dimensions.
    - Training Focus: Some models are explicitly trained on medical data, some on computer code, and others on general multi-lingual web text.
    - Sequence Length: Models have different limits on how much text they can read at once (e.g., 512 tokens versus 8,192 tokens) before creating the vector.
    


#### metadata in vector Database.
- in vector database, metadata consists of stuctured, descriptive data stored along side a vecotr embedding.
- while the vector embedding captures the general meaning or concept of data, the metadata captures the factual, rule-based details about the source document. it is usually formatted as a simple key-value pair (similar to a json object or row in a traditional database table.)
```
{
  "vector_id": "doc_chunk_8492",
  "embedding": [0.012, -0.143, 0.891, ...],
  "metadata": {
    "source_file": "2026_q2_report.pdf",
    "author": "Jane Doe",
    "page_number": 14,
    "department": "Finance",
    "is_public": true,
    "created_date": "2026-04-15"
  }
}
```

#### Why embeddings alone are not enough
- Vector embeddings are great at understanding concepts, but they fail completely at structured logic.
- An embedding cannot answer exact questions like:
    - is this information outdated?
    - Do i have permisiion to read this file.
    - which specific page did the text come from
- metadata bridges this gap. it adds structure, rules, and context to the unstructured text.

##### Every Major Use Case for Metadata.
- Metadata is essential for building production-grade AI systems, specifically Retrieval-Augmented Generation (RAG) pipelines.
1. Advanced Metadata Filtering (Pre-Filtering)
- This allows you to narrow down the search space using strict logic before the AI calculates similarity.
- Example: You search for "health benefits policy." You apply a filter where country == "India" and status == "active". The database ignores all US policies and expired files, matching only relevant Indian policies.
2. Source Attribution (Citation)
- When an LLM answers a user's question, it needs to prove where it got the information.
- Example: The metadata stores the url, document_title, and page_number. When the LLM answers a question, the application can display a clickable link: "Source: Employee Handbook, Page 4".
3. 3. Security and Access Control (RBAC)
- here we can prevent users from seeing data they are not authorized to view.
- Example: A company stores all internal documents in one vector database. When a user queries the system, the application automatically appends a filter: clearance_level <= user.clearance_level. Employees cannot accidentally retrieve confidential executive documents.
4. Time-Series Tracking (Recency) : 
- Information changes over time. Metadata helps you surface the newest information or filter out obsolete data.
- Example: You can filter search results to only look at documents where created_year >= 2025, ensuring the AI does not read outdated procedures from 2020.
5. Multi-Tenant Isolation
- If you are building a software platform (SaaS) where multiple corporate clients use your app, you must keep their data separated.
- Example: Instead of buying a separate database for every client, you can use one database and tag every vector with an organization_id. Every query is automatically locked to that specific ID.

#### Metadata Filtering
- Metadata filtering is the process of narrowing down your vector search by applying hard conditions (like dates, categories, or user IDs) before or after calculating semantic similarity.
- It combines traditional database filtering (like a SQL WHERE clause) with AI-driven similarity search.

#### Why Metadata Filtering is Necessary
- Pure similarity search only understands meaning, not strict rules.
- If you ask your vector store for "Q3 financial reports," a pure similarity search might return reports from 2022 or 2024 because they look contextually identical to a 2026 report.
- Adding metadata allows you to enforce absolute rules, such as: "Only search documents where year == 2026."


#### How it Works (The Two Approaches)
- Vector databases handle metadata filtering in one of two ways. Most modern vector stores use the second approach because it is much faster.
```
Approach 1: Post-Filtering (Filter After Search)
[Query] ──> [Semantic Search (Top 100)] ──> [Apply Metadata Filter] ──> [Final Results (May be < K)]

Approach 2: Pre-Filtering (Filter Before Search)
[Query] ──> [Apply Metadata Filter] ──> [Semantic Search on Subset] ──> [Final Results (Exactly K)]
```

1. Post-Filtering (Filter After Search)The database performs a pure similarity search first to find the top \(K\) matches. Then, it throws away any results that do not match your metadata criteria.
- The Problem: If you ask for the top 5 results, but 4 of them are filtered out by the metadata rule, you are left with only 1 result.
2. Pre-Filtering (Filter Before Search)The database uses metadata indices to instantly isolate only the records that match your criteria. Then, it runs the semantic similarity search only within that specific subset.
The Benefit: You always get the exact number of results (\(K\)) you requested, and they are guaranteed to match your rules.

### Code Example: How It Looks in Practice
- When querying a vector database, you pass the metadata filter as an extra argument alongside your search query.

```
# Example: Searching documentation but limiting by project and department
results = vector_store.similarity_search(
    query="How do I reset my API key?",
    k=3,
    filter={
        "status": "published",
        "department": "security",
        "version": {"$gte": 2.0} # Supports comparisons like greater than or equal to
    }
)
```

#### Similarity Search with Scores
- in normal search ```results = similarity_search()``` only documents returned.
- with similarity score : ```results = vectorstore.similarity_search_with_score("Sparks")```
- in output: ```[(document(...),0.89),(Document(...),0.83)]`` Score indicates relevance.

- Similarity with score is a feature in vector databases that returns a mathematical value alongside each search result to show how closely that result matches your query.
- Instead of just giving you a list of results, the database tells you exactly how confident it is in each match.

#### What the Score Represents
- The score is a number calculated by the database using its distance metric (like Cosine Similarity or Euclidean Distance).
    - High Score / Low Distance: The concept in the document is nearly identical to your query.
    - Low Score / High Distance: The concept is barely related or completely irrelevant.

##### Important Warning: Range Depends on the Metric.
- Depending on the database you use (like Pinecone, Chroma, or FAISS) and the metric you choose, the score scale will change:
    - Cosine Similarity Score: Typically ranges from 0.0 to 1.0 (or -1 to 1). A score of 1.0 means absolute identical meaning. A score of 0.0 means completely unrelated.
    - Euclidean Distance Score: Ranges from 0.0 to infinity. Because it measures distance, lower is better. A score of 0.0 means a perfect match; a score of 50.2 means the points are far apart in space.

- Code example 
```
# Querying the database for a specific topic
results = vector_store.similarity_search_with_score(
    query="What is the refund policy?", 
    k=2
)

for doc, score in results:
    print(f"Score: {score}")
    print(f"Content: {doc.page_content}\n")
```
```
Score: 0.92  
Content: "Customers can request a full refund within 30 days of purchase..."

Score: 0.41  
Content: "Our office kitchen is stocked with coffee, tea, and fresh fruit..."
```

##### Every Major Use Case for the Similarity Score
1. Setting a Quality Threshold (Cutoff Filter)
    - You can write code to automatically throw away results that fall below a certain score. This stops your AI from reading garbage information.
    - Example: If you set a threshold of 0.70, the second result in the example above (score 0.41) will be discarded and never sent to the LLM.
2. Handling "I Don't Know" Situations
- If a user asks a question completely unrelated to your business, all search scores will be very low.
    - Example: A user asks a financial bot: "How do I bake a chocolate cake?". The top vector match returns a score of 0.25. Your application can instantly catch this low score and reply: "I'm sorry, I cannot find any relevant information in the company documents to answer that."
3. Reranking for Better Accuracy
- Vector databases are built for speed, meaning their initial top 100 results are a close guess. You can use the initial scores to select a subset of data, then pass them to a slower, more precise AI model (a Reranker) to recalculate a perfect final score.
- ✅ Reranking is a two-step process where a fast vector search gathers a broad group of likely matches, and a second, smarter AI model reshuffles them to ensure the absolute most accurate answer is moved to the top before hitting the LLM.

#### MMR Search (Maximal Marginal Relevance Search).
- It is a search technique used to find results that are both highly relevant to your query and diverse from one another.
- While a standard similarity search often returns documents that repeat the exact same information, MMR forces the database to look for different perspectives and unique details. ( it reduces the redunandant information)

#### The Problem with Pure Similarity Search
- imagine you query to your database. ```Tell me about the risks of smking cigarates```
- a standard vector similarity search looks for the closest matchs in vecotr store space. it might return the top 3 results like this :
    1. Result 1: ```"Smoking cigarettes causes lung cancer."```
    2. Result 2: ```"The primary health risk of smoking is developing lung cancer."```
    3. Result 3: ```"Cigarette use heavily increases the risk of lung cancer."```
- All three answers are highly relevant, but they are redundant. They tell you the exact same thing three times. If you pass these to your LLM, it won't know about other risks like heart disease or stroke.

#### How MMR Solves This (The Balance)
- MMR optimizes for two conflicting goals at the same time:
    1. Relevance: How well does the document match the query?
    2. Diversity: How different is this document from the results we already selected?
- It calculates a blended score for each document. If a document is highly relevant but looks almost identical to a document already chosen, MMR will penalize its score and skip it, picking a slightly lower-scoring but unique document instead.
- Going back to our smoking example, an MMR search would return:
    1. Result 1: "Smoking cigarettes causes lung cancer." (High relevance)
    2. Result 2: "Nicotine use constricts blood vessels, leading to heart disease." (Slightly lower initial similarity, but highly diverse from Result 1)
    3. Result 3: "Cigarette smoke damages gum tissue and causes tooth decay." (Brings in another completely new perspective)

#### The Lambda (λ) Parameter: Controlling Diversity
- When running an MMR search, you use a parameter called Lambda (λ), which ranges from 0 to 1. It acts as a slider to control how much you value relevance versus diversity:
    - λ = 1 (Pure Similarity): Diversity is ignored completely. The algorithm behaves exactly like a standard K-Nearest Neighbors vector search.
    - λ = 0 (Pure Diversity): Relevance is ignored after the very first document. The algorithm will intentionally pick results that are as far apart from each other as possible.
    - λ = 0.5 (The Sweet Spot): A perfect balance. It selects documents that are highly relevant but ensures they don't repeat the same phrasing or ideas.

##### Code Example: 
```
results = vector_store.max_marginal_relevance_search(
    query="What are the risks of smoking cigarettes?",
    k=3,           # Number of final documents to return
    fetch_k=20,    # Number of candidate documents to fetch initially to analyze for diversity
    lambda_mult=0.5 # The lambda parameter for balancing diversity
)
```

✅ MMR (Maximal Marginal Relevance) search is an algorithm that balances text relevance with structural diversity, preventing an AI from retrieving redundant, repetitive information and ensuring a wider variety of unique facts are surfaced.

#### ANN Search Approximate Nearest Neighbor Seach:
- it is a shotcut method used by vecotr databases to search through millions of data points in milli seconds by sacrifying a tiny bit of accuracy for massive gains in speed.
- It is the core technology that allows large-scale AI applications to work in real-time.
- The Problem: Exact KNN is Too Slow:
- In the previous steps, we discussed finding the K-Nearest Neighbors (KNN). To find the exact nearest neighbors, a database must calculate the mathematical distance between your query vector and every single vector stored in the system.
    - if you have a 100 vecotrs, this is instant.
    - if you have 100m vecotrs calculating 100m distances for a single query causes massive server lag and crashes your application. this is called "brute-force"  or linear search.

##### The Solution: How ANN Search Works
- Instead of checking every single record, ANN algorithms organize the multi-dimensional vector space into smart structures (like webs, trees, or clusters) during document ingestion.
- When a query comes in, the database skips 99% of the vectors and only checks the region where the match is most likely to be. It guesses the closest neighbors approximately.
- There are three main indexing methods used to achieve this:
    1. Graph-Based (The Social Network Approach)
    - Algorithm: HNSW (Hierarchical Navigable Small World).
    - How it works: It connects vectors together like a social network. The database starts at a high-level layer with only a few far-apart points (like country-level hubs), quickly jumps to the right region, and then drops down to lower layers to find close neighbors (like street-level addresses).
    - Used by: Pinecone, Milvus, Qdrant, Chroma.
    2. Cluster-Based (The Neighborhood Approach)
    - Algorithm: IVF (Inverted File Index).
    - How it works: During ingestion, the database groups similar vectors into distinct clusters (buckets). When you query the database, it figures out which cluster center your query is closest to, and then only searches inside that specific bucket, ignoring all other clusters entirely.
    - Used by: FAISS (Meta's vector library).
    3. Tree-Based (The Decision Tree Approach)
    - Algorithm: ANNOY (Approximate Nearest Neighbors Oh Yeah).
    - How it works: It splits the vector space into geometric halves using random planes, creating a tree structure. The search query simply travels down the branches of the tree to find its closest match.


#### Speed vs. Accuracy Trade-off
Because ANN search uses shortcuts, it might occasionally miss the absolute closest vector and return the 2nd or 3rd closest instead. However, in AI applications, this minor loss in precision is invisible to the user.

| Feature | Exact KNN (Brute Force) | ANN (Approximate) |
| :--- | :--- | :--- |
| **Accuracy** | 100% Perfect | ~95% to 99% (Highly Accurate) |
| **Search Speed** | Slows down as data grows | Remains ultra-fast at scale |
| **Memory Usage** | Low | High (Indexes require RAM) |
| **Best Used For** | Small datasets (< 10k items) | Large datasets (Millions of items) |


✅ ANN (Approximate Nearest Neighbor) search is a fast indexing method that groups vectors into smart structures (like clusters or graphs), allowing the database to instantly skip irrelevant data and find close matches in milliseconds without checking every record.

In [ ]:
# Every vector store in LangChain exposes the same core interface:

# Ingestion
vectorstore = SomeVectorStore.from_documents(docs, embeddings)

# Search
vectorstore.similarity_search(query, k=4)
vectorstore.similarity_search_with_score(query, k=4)
vectorstore.max_marginal_relevance_search(query, k=4)

# As retriever (for chains)
retriever = vectorstore.as_retriever()

In [ ]:
# 1 — Chroma (best for local / development)
# pip install chromadb langchain-chroma

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Create from scratch ───────────────────────────────────
docs = [
    Document(page_content="LangChain is a framework for LLM apps.",
             metadata={"source": "intro", "chapter": 1}),
    Document(page_content="RAG retrieves relevant documents for context.",
             metadata={"source": "rag_guide", "chapter": 2}),
    Document(page_content="Chroma is a local vector database.",
             metadata={"source": "chroma_docs", "chapter": 3}),
    Document(page_content="The stock market closed higher today.",
             metadata={"source": "news", "chapter": 0}),
]

# In-memory (lost when process ends)
vectorstore = Chroma.from_documents(docs, embeddings)

# Persisted to disk (survives restarts)
vectorstore = Chroma.from_documents(
    docs,
    embeddings,
    persist_directory="./chroma_db",
    collection_name="my_collection"   # like a table name
)

# ── Search ───────────────────────────────────────────────
results = vectorstore.similarity_search("What is LangChain?", k=2)
for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

# ── Add documents later ──────────────────────────────────
new_docs = [
    Document(page_content="FAISS is a vector search library by Meta.",
             metadata={"source": "faiss_docs", "chapter": 4})
]
vectorstore.add_documents(new_docs)

# ── Delete by metadata ───────────────────────────────────
vectorstore.delete(where={"source": "news"})

ModuleNotFoundError: No module named 'langchain_chroma'

3 — Pinecone (best for production cloud scale)
- Full managed, serverless, handles billions of vectors. the go to for production deployments.

In [4]:
# pip install pinecone-client langchain-pinecone

In [18]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from sys import api_version
import os
from dotenv import load_dotenv
load_dotenv()
# create a setup api key for this 
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pc.create_index(
    name = "langchain-rag-test",
    dimension = 1536,
    metric = 'cosine',
    spec = ServerlessSpec(cloud='aws',region="us-east-1")
)

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': '6a7c310f6424b4d7c2053d9601b2c490', 'date': 'Sat, 20 Jun 2026 13:11:45 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [19]:
from langchain_core.documents import Document

# 1. Create a list of dummy Document objects
docs = [
    Document(
        page_content="LangChain is an open-source framework for building applications with LLMs.",
        metadata={"source": "tech_doc", "topic": "AI", "page": 1}
    ),
    Document(
        page_content="Python is a popular programming language known for its simple syntax and versatility.",
        metadata={"source": "coding_doc", "topic": "Programming", "page": 1}
    ),
    Document(
        page_content="Retrieval-Augmented Generation (RAG) improves LLM answers by fetching external information.",
        metadata={"source": "tech_doc", "topic": "AI", "page": 2}
    ),
    Document(
        page_content="To cook perfect pasta, boil water, add salt, and cook until al dente.",
        metadata={"source": "cooking_doc", "topic": "Recipes", "page": 1}
    )
]

# You can print the first document to verify it is structured correctly
print("Created", len(docs), "documents.")
print("Example Document:")
print("Text:", docs[0].page_content)
print("Metadata:", docs[0].metadata)


Created 4 documents.
Example Document:
Text: LangChain is an open-source framework for building applications with LLMs.
Metadata: {'source': 'tech_doc', 'topic': 'AI', 'page': 1}


In [21]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

# 1. Initialize Pinecone client
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "langchain-rag-test"

# 2. Delete the index if it already exists with the wrong dimension (1536)
if index_name in [idx.name for idx in pc.list_indexes()]:
    print(f"Deleting existing index '{index_name}'...")
    pc.delete_index(index_name)

# 3. Recreate the index with the correct dimension (384)
print(f"Creating new index '{index_name}' with dimension 384...")
pc.create_index(
    name=index_name,
    dimension=384,  # <--- Changed from 1536 to 384 to match HuggingFace
    metric='cosine',
    spec=ServerlessSpec(cloud='aws', region="us-east-1")
)
print("Index created successfully!")


Deleting existing index 'langchain-rag-test'...
Creating new index 'langchain-rag-test' with dimension 384...
Index created successfully!


In [22]:
# ingest documents 
import os
from dotenv import load_dotenv
load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name = os.getenv("huggingface_model_name"))
vectorstore = PineconeVectorStore.from_documents(
    pinecone_api_key = os.getenv("PINECONE_API_KEY"),
    documents = docs,
    embedding = embeddings,
    index_name = "langchain-rag-test",
    namespace = "__default__")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1780.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# --- search the content
results = vectorstore.similarity_search(
    "what is Langchain",
    k=4,
    name_space = "__default__"
)
print(results)

[Document(id='e0c234fb-a034-4f6d-8f6b-930651ab4783', metadata={'page': 1.0, 'source': 'tech_doc', 'topic': 'AI'}, page_content='LangChain is an open-source framework for building applications with LLMs.'), Document(id='ef4ac753-9f2e-4777-83ef-30855824a4c3', metadata={'page': 1.0, 'source': 'coding_doc', 'topic': 'Programming'}, page_content='Python is a popular programming language known for its simple syntax and versatility.'), Document(id='0017dc78-95f8-4a24-b935-89c8b59c88b1', metadata={'page': 1.0, 'source': 'cooking_doc', 'topic': 'Recipes'}, page_content='To cook perfect pasta, boil water, add salt, and cook until al dente.'), Document(id='4933e821-28e8-4a0f-b08e-0decf03fd526', metadata={'page': 2.0, 'source': 'tech_doc', 'topic': 'AI'}, page_content='Retrieval-Augmented Generation (RAG) improves LLM answers by fetching external information.')]


```
[
  Document(id='e0c234fb-a034-4f6d-8f6b-930651ab4783',
  metadata={
    'page': 1.0,
    'source': 'tech_doc',
    'topic': 'AI'
  },
  page_content='LangChain is an open-source framework for building applications with LLMs.'),
  Document(id='ef4ac753-9f2e-4777-83ef-30855824a4c3',
  metadata={
    'page': 1.0,
    'source': 'coding_doc',
    'topic': 'Programming'
  },
  page_content='Python is a popular programming language known for its simple syntax and versatility.'),
  Document(id='0017dc78-95f8-4a24-b935-89c8b59c88b1',
  metadata={
    'page': 1.0,
    'source': 'cooking_doc',
    'topic': 'Recipes'
  },
  page_content='To cook perfect pasta, boil water, add salt, and cook until al dente.'),
  Document(id='4933e821-28e8-4a0f-b08e-0decf03fd526',
  metadata={
    'page': 2.0,
    'source': 'tech_doc',
    'topic': 'AI'
  },
  page_content='Retrieval-Augmented Generation (RAG) improves LLM answers by fetching external information.')
]
```

#### Search Strategies (this is where it gets powerful)


Basic similarity search


In [24]:
results = vectorstore.similarity_search(query="what is langchain",k=4)
print(results)

[Document(id='e0c234fb-a034-4f6d-8f6b-930651ab4783', metadata={'page': 1.0, 'source': 'tech_doc', 'topic': 'AI'}, page_content='LangChain is an open-source framework for building applications with LLMs.'), Document(id='ef4ac753-9f2e-4777-83ef-30855824a4c3', metadata={'page': 1.0, 'source': 'coding_doc', 'topic': 'Programming'}, page_content='Python is a popular programming language known for its simple syntax and versatility.'), Document(id='0017dc78-95f8-4a24-b935-89c8b59c88b1', metadata={'page': 1.0, 'source': 'cooking_doc', 'topic': 'Recipes'}, page_content='To cook perfect pasta, boil water, add salt, and cook until al dente.'), Document(id='4933e821-28e8-4a0f-b08e-0decf03fd526', metadata={'page': 2.0, 'source': 'tech_doc', 'topic': 'AI'}, page_content='Retrieval-Augmented Generation (RAG) improves LLM answers by fetching external information.')]


With relevance scores


In [25]:
results = vectorstore.similarity_search_with_score(query="what is langchain",k=4)
for doc,score in results:
    print(f"Score: {score:.3f} | {doc.page_content[:80]}")

Score: 0.606 | LangChain is an open-source framework for building applications with LLMs.
Score: 0.119 | Python is a popular programming language known for its simple syntax and versati
Score: 0.102 | To cook perfect pasta, boil water, add salt, and cook until al dente.
Score: 0.009 | Retrieval-Augmented Generation (RAG) improves LLM answers by fetching external i


Metadata Filtering

In [ ]:
# onnly search in specific chapters 
results = vectorstore.similarity_search(
    query = "retrival",
    k=3,
    filter = {"chapter":2} # the 2 is pinecone index 
)
# Multiple filters (Chroma)
results = vectorstore.similarity_search(
    query="revenue growth",
    k=3,
    filter={"$and": [
        {"source": {"$eq": "annual_report"}},
        {"page":   {"$gte": 5}}
    ]}
)

MMR — Maximal Marginal Relevance
- here the problem with similarity search: if 3 of your top 4 chunks say the same thing, you have wasted context. MMR balances relevance with diversity.


In [28]:
results = vectorstore.max_marginal_relevance_search(
    query = "what is langchain?",
    k =4, # these are final number of results
    fetch_k = 20,# canditate pool to pick from 
    lambda_mult = 0.5 # 0.0 = max diversity, 1.0 = max similarity
)
print(results)
# Returns 4 results that are relevant BUT not all saying the same thing


[Document(metadata={'page': 1.0, 'source': 'tech_doc', 'topic': 'AI'}, page_content='LangChain is an open-source framework for building applications with LLMs.'), Document(metadata={'page': 1.0, 'source': 'cooking_doc', 'topic': 'Recipes'}, page_content='To cook perfect pasta, boil water, add salt, and cook until al dente.'), Document(metadata={'page': 1.0, 'source': 'coding_doc', 'topic': 'Programming'}, page_content='Python is a popular programming language known for its simple syntax and versatility.'), Document(metadata={'page': 2.0, 'source': 'tech_doc', 'topic': 'AI'}, page_content='Retrieval-Augmented Generation (RAG) improves LLM answers by fetching external information.')]


Retriever Interface
- in chain we always convert the vector store to a retriver. this gives you a clean .invoke(query) interface that chains and agents understand.

In [30]:
# Basic retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",       # or "mmr" or "similarity_score_threshold"
    search_kwargs={"k": 4}
)

# MMR retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.6}
)

# Threshold retriever — only returns results above a confidence score
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.75, "k": 6}
)

# Use it
docs = retriever.invoke("What is LangChain?")
print(docs)

[Document(id='e0c234fb-a034-4f6d-8f6b-930651ab4783', metadata={'page': 1.0, 'source': 'tech_doc', 'topic': 'AI'}, page_content='LangChain is an open-source framework for building applications with LLMs.')]


Vector stores in LangChain are specialized databases used to store embeddings and perform semantic similarity search. They store document chunks, embeddings, and metadata, enabling efficient retrieval of relevant information for RAG systems. LangChain supports vector databases such as FAISS, Chroma, Pinecone, Qdrant, Weaviate, and Milvus through a unified interface